# 面试题：为什么参数校验必须在模型之后？

可复述答案：模型输出是未可信输入，因此要在工具执行前由服务端校验语法、类型、范围、跨字段、权限、权威状态和副作用前置条件。返回结构化错误让 Agent 在有限轮数内修复；解析失败不能静默套默认值后写入。Gateway 做通用租户/风险门禁，工具服务保留不可绕过的领域校验。

## 真实案例

地址变更工具处理六个订单事件，包含国家、邮编、订单归属与发货状态。

## 基线

基线把缺失国家默认为 CN 后直接提交。

## 结果解读

手写 validator 分层输出字段和后置原因，而非仅返回一句失败文本。

## 失败案例

已经出库的订单即使邮编格式正确，也不能自动改地址。

In [1]:
events = [{'id':'A01','owner':'u1','caller':'u1','country':'CN','zip':'200120','shipped':False}, {'id':'A02','owner':'u2','caller':'u1','country':'CN','zip':'100000','shipped':False}, {'id':'A03','owner':'u3','caller':'u3','country':'CN','zip':'12','shipped':False}, {'id':'A04','owner':'u4','caller':'u4','country':'US','zip':'02139','shipped':False}, {'id':'A05','owner':'u5','caller':'u5','country':'CN','zip':'518000','shipped':True}, {'id':'A06','owner':'u6','caller':'u6','country':'','zip':'310000','shipped':False}]  # 构造六条包含身份、地址格式与订单状态的事件。
print('地址变更输入:', events)  # 输出模型生成的原始参数与权威状态快照。
print('教学说明：所有用户和订单均为离线脱敏标识，不代表真实 PII。')  # 明确数据的教学边界。

地址变更输入: [{'id': 'A01', 'owner': 'u1', 'caller': 'u1', 'country': 'CN', 'zip': '200120', 'shipped': False}, {'id': 'A02', 'owner': 'u2', 'caller': 'u1', 'country': 'CN', 'zip': '100000', 'shipped': False}, {'id': 'A03', 'owner': 'u3', 'caller': 'u3', 'country': 'CN', 'zip': '12', 'shipped': False}, {'id': 'A04', 'owner': 'u4', 'caller': 'u4', 'country': 'US', 'zip': '02139', 'shipped': False}, {'id': 'A05', 'owner': 'u5', 'caller': 'u5', 'country': 'CN', 'zip': '518000', 'shipped': True}, {'id': 'A06', 'owner': 'u6', 'caller': 'u6', 'country': '', 'zip': '310000', 'shipped': False}]
教学说明：所有用户和订单均为离线脱敏标识，不代表真实 PII。


In [2]:
def unsafe_submit(row):  # 定义缺失字段自动补默认值的错误基线。
    country = row['country'] or 'CN'  # 错误地把未知国家静默猜成中国。
    return '提交:' + country + '-' + row['zip']  # 在没有权限和状态校验时直接构造写请求。
baseline = [(row['id'], unsafe_submit(row)) for row in events]  # 对六条事件执行危险基线。
print('补默认值基线:', baseline)  # 输出看似成功却可能写错地址的结果。

补默认值基线: [('A01', '提交:CN-200120'), ('A02', '提交:CN-100000'), ('A03', '提交:CN-12'), ('A04', '提交:US-02139'), ('A05', '提交:CN-518000'), ('A06', '提交:CN-310000')]


In [3]:
def validate_address(row):  # 定义工具服务中不可绕过的分层参数验证。
    errors = []  # 初始化结构化错误列表。
    if row['caller'] != row['owner']:  # 检查调用者是否拥有订单。
        errors.append('permission:订单不属于调用者')  # 阻断跨用户地址篡改。
    if row['country'] not in {'CN','US'}:  # 检查国家枚举而不是猜默认值。
        errors.append('country:不支持或缺失')  # 指出需要追问的字段。
    if not row['zip'].isdigit() or len(row['zip']) not in {5,6}:  # 检查邮编的格式和长度。
        errors.append('zip:格式错误')  # 返回可自动修复的字段错误。
    if row['shipped']:  # 检查来自订单系统的权威发货状态。
        errors.append('state:已出库需人工处理')  # 阻断会造成物流不一致的自动写操作。
    return errors  # 返回全部错误，供上层决定追问、拒绝或升级。

In [4]:
results = [(row['id'], validate_address(row)) for row in events]  # 对六条地址变更事件执行完整校验。
print('id | 验证结果')  # 输出结构化错误表标题。
for event_id, errors in results:  # 遍历每条事件的验证结论。
    print(event_id, '可提交' if not errors else errors)  # 输出是否可提交及具体阻断原因。
print('可提交数:', sum(not errors for _, errors in results), '，需追问或升级数:', sum(bool(errors) for _, errors in results))  # 汇总可执行与阻断决策。

id | 验证结果
A01 可提交
A02 ['permission:订单不属于调用者']
A03 ['zip:格式错误']
A04 可提交
A05 ['state:已出库需人工处理']
A06 ['country:不支持或缺失']
可提交数: 2 ，需追问或升级数: 4


In [5]:
wrong = unsafe_submit(events[4])  # 演示已出库订单在默认值基线中仍会被提交。
fixed = validate_address(events[4])  # 使用权威状态校验同一请求。
print('失败案例 A05：基线=', wrong, '，修正=', fixed)  # 展示格式正确不代表业务前置条件成立。
print('生产差距：真实系统需要 schema 版本、错误码、有限修复轮数、审计日志与 Gateway 加服务端双层门禁。')  # 说明验证器的生产扩展点。

失败案例 A05：基线= 提交:CN-518000 ，修正= ['state:已出库需人工处理']
生产差距：真实系统需要 schema 版本、错误码、有限修复轮数、审计日志与 Gateway 加服务端双层门禁。


In [6]:
assert validate_address(events[0]) == []  # 验证合法且未出库的自有订单可提交。
assert 'permission:订单不属于调用者' in validate_address(events[1])  # 验证跨用户请求会被拒绝。
assert 'state:已出库需人工处理' in fixed  # 验证已出库反例会升级而非写入。